In [1]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# =========================
# PATHS
# =========================

base = "/kaggle/input/datasets/carterrishi/phenobench/PhenoBench"

splits = ["train", "val"]

out_base = "/kaggle/working/dataset_yolo"

# =========================
# SEMANTIC MAPPING
# =========================

# crop = 1 + 3
# weed = 2 + 4

def get_class_from_semantic(values):

    crop = np.sum((values == 1) | (values == 3))
    weed = np.sum((values == 2) | (values == 4))

    if crop >= weed:
        return 0  # crop
    else:
        return 1  # weed


# =========================
# CONVERSION LOOP
# =========================

for split in splits:

    img_dir = os.path.join(base, split, "images")
    inst_dir = os.path.join(base, split, "plant_instances")
    sem_dir  = os.path.join(base, split, "semantics")

    out_img_dir = os.path.join(out_base, "images", split)
    out_lbl_dir = os.path.join(out_base, "labels", split)

    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)

    files = sorted(os.listdir(img_dir))

    for f in tqdm(files):

        img_path = os.path.join(img_dir, f)
        inst_path = os.path.join(inst_dir, f)
        sem_path  = os.path.join(sem_dir, f)

        img = cv2.imread(img_path)
        inst = cv2.imread(inst_path, cv2.IMREAD_UNCHANGED)
        sem  = cv2.imread(sem_path, cv2.IMREAD_UNCHANGED)

        if len(inst.shape) == 3:
            inst = inst[:, :, 0]
        if len(sem.shape) == 3:
            sem = sem[:, :, 0]

        h, w = inst.shape

        ids = np.unique(inst)
        ids = ids[ids != 0]

        lines = []

        for inst_id in ids:

            mask = (inst == inst_id)

            sem_values = sem[mask]

            if len(sem_values) == 0:
                continue

            cls = get_class_from_semantic(sem_values)

            # contour
            cnts, _ = cv2.findContours(
                mask.astype(np.uint8),
                cv2.RETR_EXTERNAL,
                cv2.CHAIN_APPROX_SIMPLE
            )

            if len(cnts) == 0:
                continue

            # =========================
            # CONNECT FRAGMENTS
            # =========================
            
            kernel = np.ones((20, 20), np.uint8)
            
            connected = cv2.morphologyEx(
                mask.astype(np.uint8),
                cv2.MORPH_CLOSE,
                kernel
            )
            
            # neue Konturen nach Verbindung
            cnts, _ = cv2.findContours(
                connected,
                cv2.RETR_EXTERNAL,
                cv2.CHAIN_APPROX_SIMPLE
            )
            
            if len(cnts) == 0:
                continue
            
            # jetzt größte äußere Form nehmen
            cnt = max(cnts, key=cv2.contourArea)
            #cnt = max(cnts, key=cv2.contourArea)
            cnt = cnt.squeeze()

            if len(cnt.shape) != 2:
                continue

            coords = []

            for x, y in cnt:
                coords.append(x / w)
                coords.append(y / h)

            line = str(cls) + " " + " ".join(map(str, coords))
            lines.append(line)

        # save label
        out_label_path = os.path.join(
            out_lbl_dir,
            f.replace(".png", ".txt")
        )

        with open(out_label_path, "w") as f_out:
            f_out.write("\n".join(lines))

        # copy image
        cv2.imwrite(os.path.join(out_img_dir, f), img)

print("DONE")

100%|██████████| 772/772 [02:24<00:00,  5.34it/s]

DONE
